In [2]:
import sys
from pathlib import Path

def encontrar_raiz_repo(marcador=".git"):
    caminho = Path.cwd()
    for pasta in [caminho, *caminho.parents]:
        if (pasta / marcador).exists():
            return pasta
    raise FileNotFoundError(f"Não achei {marcador} subindo a partir de {caminho}")

RAIZ = encontrar_raiz_repo()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.clustering import Clustering
import pandas as pd

## Carregando os dados

In [3]:
df_train = pd.read_csv(RAIZ / "data/processed/df_estruturada/df_train.csv", index_col=0)
df_valid = pd.read_csv(RAIZ / "data/processed/df_estruturada/df_valid.csv", index_col=0)
df_test = pd.read_csv(RAIZ / "data/processed/df_estruturada/df_test.csv", index_col=0)

print(df_train.shape)
df_train.head()

(4926, 11)


,pct_genes_cromossomal_efflux,mdr_score_cromossomal,mdr_score_plasmidial,pct_contigs_plasmidial,pct_plasmidial_amr,pct_plasmidial_conjugacao_amr,n_proviruses_por_mb,gc_diff_plasmid_cromossomo,virulence_score_cromossomal,virulence_score_plasmidial,cat_plasmidial_beta-lactam
sample,,,,,,,,,,,
GCA_021009385.1,7.142857,13,3,11.818182,30.769231,0.000000,1.662461,0.33,6,4,0
GCA_015825355.1,12.500000,3,0,1.666667,0.000000,0.000000,0.942911,-6.27,6,0,0
GCA_038594105.1,1.785714,12,7,14.925373,10.000000,3.333333,0.842382,-1.08,8,3,1
GCA_019201995.1,1.785714,14,5,21.487603,7.692308,0.000000,1.323753,-1.00,8,2,0
GCA_038066875.1,25.000000,4,3,25.668449,4.166667,0.000000,0.841943,-0.31,3,1,0


In [4]:
metadata_cols = pd.read_csv(RAIZ / "data/processed/metadata_filtrada.csv")

metadata_cols = metadata_cols.set_index('sample')
metadata_cols.head()

,Species,Source,Date,Location,BioSample,Estado,Região,source_type,WHO_Priority
sample,,,,,,,,,
GCA_000216055.2,Leptospira interrogans,Homo sapiens,NaN,Salvador,SAMN00254327,BA,Nordeste,Other,Other
GCA_000316425.1,Escherichia coli,NaN,1990.0,Brazil,SAMN01041333,NaN,NaN,Unknown,Critical
GCA_000223095.2,Vibrio cholerae,patient with cholera-like diarrhea,1991.0,NaN,SAMN02470783,NaN,NaN,Gastrointestinal,Other
GCA_036761135.1,Acinetobacter bereziniae,Rectal Swab,2019.0,Brazil,SAMN39408214,NaN,NaN,Gastrointestinal,Other
GCA_003670255.1,Acinetobacter bereziniae,Endotracheal aspirate,2014.0,"Londrina, PR",SAMN09907131,PR,Sul,Respiratory,Other


In [5]:
metadata_alinhado = metadata_cols.loc[df_train.index]

print(metadata_alinhado.shape)
metadata_alinhado.head()

(4926, 9)


,Species,Source,Date,Location,BioSample,Estado,Região,source_type,WHO_Priority
sample,,,,,,,,,
GCA_021009385.1,Staphylococcus aureus,Nasal Colonization,2016.0,Rio de Janeiro,SAMN16946800,RJ,Sudeste,Respiratory,High
GCA_015825355.1,Neisseria meningitidis,cerebrospinal fluid,2016.0,"Sao Paulo, Sao Paulo",SAMN10873099,SP,Sul,CNS,Other
GCA_038594105.1,Pseudomonas aeruginosa,Surveillance swab,2023.0,"Sao Paulo, Sao Paulo",SAMN40996642,SP,Sul,Other,High
GCA_019201995.1,Escherichia coli,urine,2014.0,Uberlandia,SAMN14464882,NaN,NaN,Urinary,Critical
GCA_038066875.1,Burkholderia contaminans,blood,2023.0,Sao Paulo,SAMN40603088,SP,Sul,Blood,Other


In [6]:
top_especies = metadata_alinhado["Species"].value_counts().nlargest(10).index

especie_plot_completo = metadata_alinhado["Species"].where(
    metadata_alinhado["Species"].isin(top_especies), "Outras"
)

especie_plot_completo.head()

sample
GCA_021009385.1     Staphylococcus aureus
GCA_015825355.1    Neisseria meningitidis
GCA_038594105.1    Pseudomonas aeruginosa
GCA_019201995.1          Escherichia coli
GCA_038066875.1                    Outras
Name: Species, dtype: object

In [7]:
colunas_X = ['pct_genes_cromossomal_efflux', 'mdr_score_cromossomal',
       'mdr_score_plasmidial', 'pct_contigs_plasmidial', 'pct_plasmidial_amr',
       'pct_plasmidial_conjugacao_amr', 'n_proviruses_por_mb',
       'gc_diff_plasmid_cromossomo', 'virulence_score_cromossomal',
       'virulence_score_plasmidial', 'cat_plasmidial_beta-lactam']

In [8]:
colunas_X_menor = ['pct_genes_cromossomal_efflux', 'mdr_score_cromossomal',
       'mdr_score_plasmidial', 'pct_plasmidial_amr',
       'virulence_score_plasmidial', 'cat_plasmidial_beta-lactam']

In [9]:
transformacoes = ["sem_scaling", "standard", "minmax"]

## K-means

In [10]:
cluster_kmeans = Clustering(colunas_X, df_train, "kmeans")
tabela_cv_kmeans = cluster_kmeans.cross_validation(transformacao="sem_scaling")

In [11]:
tabela_cv_kmeans

,n_clusters,init,n_init,mean_score,std_score,silhouette,calinski_harabasz,davies_bouldin
0,2,k-means++,auto,0.412610,0.018577,0.414,2760.048,1.101
1,2,k-means++,10,0.412715,0.022502,0.414,2760.053,1.101
2,2,k-means++,20,0.411591,0.022048,0.414,2760.053,1.101
3,2,random,auto,0.411574,0.022075,0.414,2760.053,1.101
4,2,random,10,0.411574,0.022075,0.414,2760.053,1.101
5,2,random,20,0.411574,0.022075,0.414,2760.053,1.101
6,3,k-means++,auto,0.389614,0.006996,0.391,2776.276,1.045
7,3,k-means++,10,0.391632,0.006401,0.390,2778.658,1.052
8,3,k-means++,20,0.391604,0.006382,0.390,2778.658,1.052
9,3,random,auto,0.390241,0.005989,0.390,2778.658,1.052


In [15]:
metricas = {
    "silhouette": "max",
    "calinski_harabasz": "max",
    "davies_bouldin": "min"
}

def destacar_top2(s, tipo):
    if tipo == "max":
        indices = s.nlargest(2).index
    else:
        indices = s.nsmallest(2).index

    return [
        "background-color: red" if i in indices else ""
        for i in s.index
    ]

tabela_cv_kmeans.style\
    .apply(
        lambda s: destacar_top2(s, metricas[s.name]),
        subset=list(metricas.keys())
    )\
    .format({
        "silhouette": "{:.3f}",
        "calinski_harabasz": "{:.3f}",
        "davies_bouldin": "{:.3f}"
    })

,n_clusters,init,n_init,mean_score,std_score,silhouette,calinski_harabasz,davies_bouldin
0,2,k-means++,auto,0.412610,0.018577,0.414,2760.048,1.101
1,2,k-means++,10,0.412715,0.022502,0.414,2760.053,1.101
2,2,k-means++,20,0.411591,0.022048,0.414,2760.053,1.101
3,2,random,auto,0.411574,0.022075,0.414,2760.053,1.101
4,2,random,10,0.411574,0.022075,0.414,2760.053,1.101
5,2,random,20,0.411574,0.022075,0.414,2760.053,1.101
6,3,k-means++,auto,0.389614,0.006996,0.391,2776.276,1.045
7,3,k-means++,10,0.391632,0.006401,0.390,2778.658,1.052
8,3,k-means++,20,0.391604,0.006382,0.390,2778.658,1.052
9,3,random,auto,0.390241,0.005989,0.390,2778.658,1.052


In [17]:
cluster_kmeans = Clustering(colunas_X, df_train, "kmeans")
tabela_cv_kmeans_mm = cluster_kmeans.cross_validation(transformacao="minmax")

tabela_cv_kmeans_mm.style\
    .apply(
        lambda s: destacar_top2(s, metricas[s.name]),
        subset=list(metricas.keys())
    )\
    .format({
        "silhouette": "{:.3f}",
        "calinski_harabasz": "{:.3f}",
        "davies_bouldin": "{:.3f}"
    })

/usr/local/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,n_clusters,init,n_init,mean_score,std_score,silhouette,calinski_harabasz,davies_bouldin
0,2,k-means++,auto,0.445401,0.005570,0.446,4895.252,0.908
1,2,k-means++,10,0.445434,0.005617,0.446,4895.252,0.908
2,2,k-means++,20,0.445434,0.005617,0.446,4895.252,0.908
3,2,random,auto,0.445434,0.005617,0.446,4895.252,0.908
4,2,random,10,0.445434,0.005617,0.446,4895.252,0.908
5,2,random,20,0.445434,0.005617,0.446,4895.252,0.908
6,3,k-means++,auto,0.361819,0.013114,0.364,3611.313,1.149
7,3,k-means++,10,0.365644,0.005583,0.364,3611.313,1.149
8,3,k-means++,20,0.365853,0.005878,0.364,3611.313,1.149
9,3,random,auto,0.365616,0.005544,0.364,3611.313,1.149
